# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [13]:
# Print number of rows and columns
print("Shape of DataFrame:", spaceship.shape)  # `.shape` returns (rows, columns) :contentReference[oaicite:13]{index=13}


Shape of DataFrame: (8693, 14)


**Check for data types**

In [14]:
# Display each column’s dtype and the count of non-null entries
spaceship.info()  # Shows dtypes (object, float64, bool, etc.) and null counts :contentReference[oaicite:15]{index=15}


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


**Check for missing values**

In [15]:
# Count missing (NaN) values in each column
print("Missing values per column:\n", spaceship.isnull().sum())  # `.isnull().sum()` tallies nulls per column :contentReference[oaicite:17]{index=17}


Missing values per column:
 PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [16]:
# Remove all rows that have at least one missing value
spaceship_dropna = spaceship.dropna(axis=0, how="any")  # Drops rows with any NaN :contentReference[oaicite:20]{index=20}

# Verify new shape
print("New shape after dropna:", spaceship_dropna.shape)  # Confirms number of rows left after dropping nulls :contentReference[oaicite:21]{index=21}


New shape after dropna: (6606, 14)


- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [17]:
# Extract deck letter from Cabin (everything before the first '/')
spaceship_dropna["Deck"] = spaceship_dropna["Cabin"].str.split("/", expand=True)[0]  # Splits on '/' and selects index 0 :contentReference[oaicite:24]{index=24}

# Verify the unique decks
print("Unique deck values:", spaceship_dropna["Deck"].unique())  # Should show ['B', 'F', 'A', 'G', 'C', 'E', 'D', 'T'] :contentReference[oaicite:25]{index=25}


Unique deck values: ['B' 'F' 'A' 'G' 'E' 'C' 'D' 'T']


C:\Users\michele\AppData\Local\Temp\ipykernel_29700\2057126079.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  spaceship_dropna["Deck"] = spaceship_dropna["Cabin"].str.split("/", expand=True)[0]  # Splits on '/' and selects index 0 :contentReference[oaicite:24]{index=24}


- Drop PassengerId and Name

In [18]:
# Drop columns that are unique identifiers or not useful
spaceship_dropna = spaceship_dropna.drop(columns=["PassengerId", "Name", "Cabin"])  # Also drop original Cabin :contentReference[oaicite:29]{index=29}

# Confirm columns removed
print("Remaining columns:", spaceship_dropna.columns.tolist())


Remaining columns: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported', 'Deck']


- For non-numerical columns, do dummies.

In [19]:
# Identify which columns are still object or boolean (categorical)
cat_cols = spaceship_dropna.select_dtypes(include=["object", "bool"]).columns.tolist()
print("Categorical columns to encode:", cat_cols)  # e.g., ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Transported', 'Deck']

# Exclude target 'Transported' from encoding here—handle it separately
cat_cols.remove("Transported")

# Perform one-hot encoding on all categorical features except target
spaceship_dummies = pd.get_dummies(
    spaceship_dropna, columns=cat_cols, drop_first=True
)  # `drop_first=True` avoids multicollinearity :contentReference[oaicite:32]{index=32}

# Verify the new shape and columns
print("After dummy encoding, shape:", spaceship_dummies.shape)
print("Sample columns:", spaceship_dummies.columns[:10].tolist())


Categorical columns to encode: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Transported', 'Deck']
After dummy encoding, shape: (6606, 20)
Sample columns: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported', 'HomePlanet_Europa', 'HomePlanet_Mars', 'CryoSleep_True']


**Perform Train Test Split**

In [20]:
# Target variable: Transported → convert boolean to integer
y = spaceship_dummies["Transported"].astype(int)  # True→1, False→0 :contentReference[oaicite:35]{index=35}

# Features: drop the target column
X = spaceship_dummies.drop(columns=["Transported"])

print("Final feature matrix shape:", X.shape, "| Target vector length:", y.shape)  # Confirms dimensions match :contentReference[oaicite:36]{index=36}


Final feature matrix shape: (6606, 19) | Target vector length: (6606,)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [21]:
# Stratify by y to maintain the same transported/non-transported ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)  # `stratify=y` ensures balanced classes 

print("X_train shape:", X_train.shape, "| X_test shape:", X_test.shape)  # E.g., (12000, N_features) and (3000, N_features) :contentReference[oaicite:39]{index=39}
print("y_train distribution:", np.bincount(y_train), "| y_test distribution:", np.bincount(y_test))  # Shows class balance 


X_train shape: (5284, 19) | X_test shape: (1322, 19)
y_train distribution: [2623 2661] | y_test distribution: [656 666]


- Evaluate your model's performance. Comment it

In [22]:
from sklearn.neighbors import KNeighborsClassifier  # KNN classifier API 
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)  # Metrics for classification 

# Initialize KNN with default n_neighbors=5
knn = KNeighborsClassifier()  # Default hyperparameter values 

# Fit the model on the training data
knn.fit(X_train, y_train)  # KNN memorizes training instances to compute nearest neighbors 

# Predict on the test set
y_pred = knn.predict(X_test)  # For each X_test sample, finds k nearest neighbors and votes on class 


c:\Users\michele\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\michele\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\michele\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\michele\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\michele\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi

In [23]:
# Calculate overall accuracy
acc = accuracy_score(y_test, y_pred)  # Correct predictions / total predictions 
print("Test Accuracy:", acc)

# Detailed classification metrics
print("\nClassification Report:\n", classification_report(y_test, y_pred))  # Precision, recall, F1-score per class 

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)  # Rows = actual classes, columns = predicted classes 
print("\nConfusion Matrix:\n", cm)


Test Accuracy: 0.7685325264750378

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.76      0.76       656
           1       0.77      0.78      0.77       666

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322


Confusion Matrix:
 [[497 159]
 [147 519]]
